# LinguaFranca — Phase 2: Probe Training on TF-IDF Distractor Dataset (Fixed)

This notebook trains linear probes on the **distractor-RAG hidden states** and reports
the full, corrected suite of metrics for the paper:

### Fixes & Methodology:
1. **Label Semantics**: `label = 0` (Success / Entity Matched), `label = 1` (Failure / Hallucination).
2. **Explicit F1 Reporting**: Both `F1(failure, pos_label=1)` and `F1(success, pos_label=0)` reported.
3. **True Reasoning Failures Populated**: Reads `gold_entities_in_context` per-hop (`gold in context AND label=1`).
4. **Valid Shuffled Control**: Evaluates **AUROC** on shuffled training labels (expected ~0.50), not deceptive accuracy.
5. **Hop-Index Diagnostic**: Separates and filters hops > 2 (`max_hop=2`) to prevent artificial failure rate inflation.
6. **Evidence-Presence Baseline**: Predicts success if gold entity is in context, failure otherwise.
7. **Cross-Fold Variance Error Bars**: 5-fold grouped CV spread used for error margins across layers 8–14.
8. **Hop-Order Retrieval Conditions**: `only_hop1_passage` and `only_hop2_passage` (question-entry aware).
9. **Automatic HuggingFace Caching**: Uploads `labeled.jsonl` and `wikidata_aliases.json` to HF at the end.

**Hardware**: CPU-only (~5–10 min if cached; ~1 hr if downloading Wikidata aliases for the first time).

## Step 0 — Install & Clone

In [ ]:
%%capture
!pip install scikit-learn datasets huggingface_hub joblib tqdm scipy matplotlib -q


In [ ]:
import os, sys, subprocess
REPO_URL = 'https://github.com/ANLPproject/LinguaFranca.git'
BRANCH   = 'experiment/distractor-rag'
WORK_DIR = '/kaggle/working/LinguaFranca'
if not os.path.exists(WORK_DIR):
    print(f'Cloning branch {BRANCH}...')
    subprocess.run(['git', 'clone', '-b', BRANCH, REPO_URL, WORK_DIR], check=True)
else:
    print(f'Pulling latest {BRANCH}...')
    subprocess.run(['git', '-C', WORK_DIR, 'fetch', 'origin'], check=True)
    subprocess.run(['git', '-C', WORK_DIR, 'checkout', BRANCH], check=True)
    subprocess.run(['git', '-C', WORK_DIR, 'pull', 'origin', BRANCH], check=True)
os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
print('Branch:', subprocess.run(['git','branch','--show-current'], capture_output=True, text=True).stdout.strip())


## Step 1 — Credentials & Config

In [ ]:
import os
HF_TOKEN        = 'hf_your_token_here'           # paste your read/write token
HF_REPO_ID      = 'AnishRacherla/LinguaFranca-Phase3'
HF_SUBFOLDER    = 'tfidf_distractor_rag'
HF_REVISION     = 'distractor-rag-v2'
PROBE_SUBFOLDER = 'probes/tfidf_distractor_rag'

os.environ['HF_TOKEN']   = HF_TOKEN
os.environ['HF_REPO_ID'] = HF_REPO_ID
from huggingface_hub import whoami
try:
    u = whoami(token=HF_TOKEN)
    print(f"Logged in as: {u['name']} | Repo: {HF_REPO_ID} (Branch: {HF_REVISION})")
except Exception as e:
    print('HF auth warning/error:', e)


## Step 2 — Download Dataset, Hidden States & Cached Files from HF

In [ ]:
import shutil, json
from pathlib import Path
from huggingface_hub import hf_hub_download, snapshot_download

prefix   = f'{HF_SUBFOLDER}/'
revision = HF_REVISION

cot_local     = Path('/kaggle/working/data/raw/2wikimultihopqa/generated_cot.jsonl')
labeled_local = Path('/kaggle/working/data/raw/2wikimultihopqa/labeled.jsonl')
aliases_local = Path('/kaggle/working/data/raw/wikidata_aliases.json')
hs_dir        = Path('/kaggle/working/data/hidden_states')
hs_dir.mkdir(parents=True, exist_ok=True)
cot_local.parent.mkdir(parents=True, exist_ok=True)

def try_hf_download(hf_path, local_path, desc):
    if local_path.exists():
        print(f'{desc}: already on disk')
        return True
    try:
        p = hf_hub_download(
            repo_id=HF_REPO_ID, filename=hf_path, revision=revision,
            repo_type='dataset', token=HF_TOKEN,
            local_dir='/kaggle/working/hf_tmp'
        )
        shutil.copy(p, local_path)
        print(f'{desc}: downloaded from HF -> {local_path}')
        return True
    except Exception as e:
        print(f'{desc}: not on HF ({e})')
        return False

# 1. Download generated_cot.jsonl
try_hf_download(f'{prefix}data/raw/2wikimultihopqa/generated_cot.jsonl', cot_local, 'generated_cot.jsonl')

# 2. Try to download already-labeled files (saves 1 hr if previously uploaded)
try_hf_download(f'{prefix}data/raw/2wikimultihopqa/labeled.jsonl', labeled_local, 'labeled.jsonl')
try_hf_download('wikidata_aliases.json', aliases_local, 'wikidata_aliases.json')

# 3. Download hidden states
existing_pts = list(hs_dir.glob('*.pt'))
print(f'\nHidden states on disk: {len(existing_pts)}')
print(f'Syncing from {HF_REPO_ID} [{revision}]...')
try:
    snap = snapshot_download(
        repo_id=HF_REPO_ID, revision=revision, repo_type='dataset',
        allow_patterns=[f'{prefix}data/hidden_states/*.pt'],
        local_dir='/kaggle/working/hf_hs_sync', token=HF_TOKEN
    )
    src_dir = Path(f'/kaggle/working/hf_hs_sync/{prefix}data/hidden_states')
    copied = 0
    for pt in src_dir.glob('*.pt'):
        dest = hs_dir / pt.name
        if not dest.exists():
            shutil.copy(pt, dest)
            copied += 1
    print(f'  -> Copied {copied} new .pt files. Total: {len(list(hs_dir.glob("*.pt")))}')
except Exception as e:
    print(f'HF sync notice: {e}')
    print(f'Continuing with {len(list(hs_dir.glob("*.pt")))} local .pt files')


## Step 3 — Label Hops (if not already cached)

In [ ]:
import yaml, json
from pathlib import Path

labeled_local = Path('/kaggle/working/data/raw/2wikimultihopqa/labeled.jsonl')
aliases_local = Path('/kaggle/working/data/raw/wikidata_aliases.json')

if not labeled_local.exists():
    print('Running label_hops on generated_cot.jsonl...')
    with open('configs/data_config.yaml') as f:
        cfg = yaml.safe_load(f)
    cfg['data']['raw_dir']           = '/kaggle/working/data/raw'
    cfg['data']['hidden_states_dir'] = '/kaggle/working/data/hidden_states'
    cfg['matching']['wikidata_aliases_path'] = str(aliases_local)
    with open('configs/data_config.yaml', 'w') as f:
        yaml.dump(cfg, f)
    from phase1_dataset.label_hops import run_labeling
    run_labeling(cfg)
    print('label_hops complete!')
else:
    print('labeled.jsonl already on disk')

with open(labeled_local) as f:
    examples = [json.loads(l) for l in f if l.strip()]

hs_dir   = Path('/kaggle/working/data/hidden_states')
valid_ex = [ex for ex in examples if (hs_dir / f"{ex['id']}.pt").exists()]
print(f'Total labeled: {len(examples)} | With hidden states: {len(valid_ex)}')

from collections import Counter
hop1_labels = [h['label'] for ex in valid_ex
               for h in ex.get('hops', [])
               if h.get('hop_idx') == 1 and h.get('label') in (0, 1)]
print(f'Hop-1 label counts (0=Success, 1=Failure): {Counter(hop1_labels)}')
rc_counts = Counter(ex.get('retrieval_condition', 'unknown') for ex in valid_ex)
print(f'Retrieval condition breakdown: {rc_counts}')

# Hop-index diagnostic: count hops per index
hop_counts = Counter(
    h['hop_idx'] for ex in valid_ex
    for h in ex.get('hops', []) if h.get('label') in (0, 1)
)
print(f'\nHop-index distribution across all examples: {dict(sorted(hop_counts.items()))}')
extra = sum(v for k, v in hop_counts.items() if k > 2)
total = sum(hop_counts.values())
print(f'Hops beyond hop-2 (excluded from analysis): {extra}/{total} ({100*extra/total:.1f}%)')


## Step 4 — Train / Test Split

In [ ]:
import numpy as np, torch
from sklearn.model_selection import GroupShuffleSplit

groups = [ex.get('clean_pair_id', ex['id']) for ex in valid_ex]
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(valid_ex, groups=groups))
train_ex = [valid_ex[i] for i in train_idx]
test_ex  = [valid_ex[i] for i in test_idx]
print(f'Train questions: {len(train_ex)} | Test questions: {len(test_ex)}')

sample_pt = torch.load(hs_dir / f"{valid_ex[0]['id']}.pt", weights_only=False)
layer_indices = sample_pt.get('layer_indices', list(range(sample_pt['pooled'].shape[0])))
n_layers = len(layer_indices)
print(f'Layer indices ({n_layers} total): {layer_indices[:5]}...{layer_indices[-3:]}')


## Step 5 — Feature Extraction Helpers

In [ ]:
import numpy as np, torch

def extract_data(ex_list, hs_dir, layer_idx, hop_filter=None, max_hop=2):
    """
    Extract pooled hidden states and annotations.
    hop_filter: None = all hops (up to max_hop), 0 = hop-1 only, 1 = hop-2 only
    max_hop: exclude hops at/beyond this 0-indexed value. Default 2 = keep hop-1 and hop-2.
    Returns:
      X (np.ndarray), y (np.ndarray: 0=success, 1=failure),
      hop_idxs, texts, gold_ents, ret_conds, ex_ids, gold_in_ctx_flags
    """
    X, y, hop_idxs, texts, gold_ents, ret_conds, ex_ids, gold_in_ctx = [], [], [], [], [], [], [], []
    for ex in ex_list:
        pt_path = hs_dir / f"{ex['id']}.pt"
        if not pt_path.exists(): continue
        data   = torch.load(pt_path, weights_only=False)
        pooled = data['pooled']  # [n_layers, n_hops, hidden_dim]
        ex_gic = ex.get('gold_entities_in_context', [])
        for hop in ex.get('hops', []):
            h_idx = hop['hop_idx'] - 1  # 0-indexed
            if hop.get('label') not in (0, 1): continue
            if h_idx >= max_hop: continue
            if hop_filter is not None and h_idx != hop_filter: continue
            if h_idx >= pooled.shape[1]: continue
            vec = pooled[layer_idx, h_idx].detach().numpy()
            X.append(np.nan_to_num(vec))
            y.append(hop['label'])  # 0=success, 1=failure
            hop_idxs.append(h_idx)
            texts.append(hop.get('text', ''))
            gold_ents.append(hop.get('bridging_entity_gold', ''))
            ret_conds.append(ex.get('retrieval_condition', 'unknown'))
            ex_ids.append(ex['id'])
            gold_in_ctx.append(ex_gic[h_idx] if h_idx < len(ex_gic) else None)
    dim = pooled.shape[2] if X else 1
    return (
        np.array(X) if X else np.zeros((0, dim)),
        np.array(y), np.array(hop_idxs),
        texts, gold_ents, ret_conds, ex_ids, gold_in_ctx
    )

print('Feature extraction helpers ready. max_hop=2 filters out spurious extra hops.')


## Step 6 — Baselines & Probe Training (All Layers)

In [ ]:
import joblib, json
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.feature_extraction.text import TfidfVectorizer
from pathlib import Path

probe_dir = Path('/kaggle/working/probes_tfidf_rag')
probe_dir.mkdir(exist_ok=True)

_, y_tr_all, hop_idxs_tr, texts_tr, _, _, _, _ = extract_data(train_ex, hs_dir, layer_idx=layer_indices[0])
_, y_te_all, hop_idxs_te, texts_te, golds_te_all, rc_te_all, _, gold_ctx_te_all = extract_data(test_ex, hs_dir, layer_idx=layer_indices[0])

# ── HOP-INDEX DIAGNOSTIC ──────────────────────────────────────────────────────
print('=' * 100)
print('HOP-INDEX FAILURE RATE DIAGNOSTIC (Gold Hops 1 & 2)')
print('=' * 100)
for hi in [0, 1]:
    mask = hop_idxs_te == hi
    if mask.sum() == 0: continue
    fail_rate = (y_te_all[mask] == 1).mean()  # label 1 = failure
    succ_rate = (y_te_all[mask] == 0).mean()  # label 0 = success
    print(f'  hop-{hi+1}: N={mask.sum()} | Failure rate (label=1): {fail_rate*100:.1f}% | Success rate: {succ_rate*100:.1f}%')

# ── BASELINES ─────────────────────────────────────────────────────────────────
print('\n' + '=' * 100)
print('BASELINES (Test set, hops 1-2 only)')
print('Note: F1_fail = pos_label=1 (failure class); F1_succ = pos_label=0 (success class)')
print('=' * 100)

# 1. Majority class baseline
maj_class = int((y_tr_all == 1).mean() >= 0.5)  # majority class in train
majority_pred = np.full(len(y_te_all), maj_class)
f1_maj_fail = f1_score(y_te_all, majority_pred, pos_label=1, zero_division=0)
f1_maj_succ = f1_score(y_te_all, majority_pred, pos_label=0, zero_division=0)
print(f'MAJORITY CLASS BASELINE (class={maj_class}) -> Acc: {accuracy_score(y_te_all, majority_pred):.3f} | F1_fail: {f1_maj_fail:.3f} | F1_succ: {f1_maj_succ:.3f}')

# 2. TF-IDF text baseline
tfidf_vec = TfidfVectorizer(max_features=1000)
X_tr_txt  = tfidf_vec.fit_transform(texts_tr)
X_te_txt  = tfidf_vec.transform(texts_te)
clf_txt   = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf_txt.fit(X_tr_txt, y_tr_all)
y_pred_txt = clf_txt.predict(X_te_txt)
y_prob_txt = clf_txt.predict_proba(X_te_txt)[:, 1]  # P(label=1 / failure)
f1_txt_fail = f1_score(y_te_all, y_pred_txt, pos_label=1, zero_division=0)
f1_txt_succ = f1_score(y_te_all, y_pred_txt, pos_label=0, zero_division=0)
print(f'TF-IDF TEXT BASELINE         -> Acc: {accuracy_score(y_te_all, y_pred_txt):.3f} | F1_fail: {f1_txt_fail:.3f} | F1_succ: {f1_txt_succ:.3f} | AUROC: {roc_auc_score(y_te_all, y_prob_txt):.3f}')

# 3. Hop-index baseline
max_h = int(hop_idxs_te.max()) + 1
X_tr_hop = np.eye(max_h)[hop_idxs_tr.astype(int)]
X_te_hop = np.eye(max_h)[hop_idxs_te.astype(int)]
clf_hop  = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
clf_hop.fit(X_tr_hop, y_tr_all)
y_pred_hop = clf_hop.predict(X_te_hop)
y_prob_hop = clf_hop.predict_proba(X_te_hop)[:, 1]
f1_hop_fail = f1_score(y_te_all, y_pred_hop, pos_label=1, zero_division=0)
f1_hop_succ = f1_score(y_te_all, y_pred_hop, pos_label=0, zero_division=0)
print(f'HOP-INDEX BASELINE           -> Acc: {accuracy_score(y_te_all, y_pred_hop):.3f} | F1_fail: {f1_hop_fail:.3f} | F1_succ: {f1_hop_succ:.3f} | AUROC: {roc_auc_score(y_te_all, y_prob_hop):.3f}')

# 4. Evidence-presence baseline (Hop-1 only)
h1_te_mask = hop_idxs_te == 0
gic_h1 = [gold_ctx_te_all[i] for i in range(len(gold_ctx_te_all)) if h1_te_mask[i]]
y_te_h1_b = y_te_all[h1_te_mask]
known = np.array([g is not None for g in gic_h1])
if known.sum() > 0:
    # If entity is in context (g==True), predict Success (0); if missing, predict Failure (1)
    y_pred_evid = np.array([0 if g else 1 for g in gic_h1])
    f1_ev_fail = f1_score(y_te_h1_b[known], y_pred_evid[known], pos_label=1, zero_division=0)
    f1_ev_succ = f1_score(y_te_h1_b[known], y_pred_evid[known], pos_label=0, zero_division=0)
    print(f'EVIDENCE-PRESENCE (Hop-1)    -> Acc: {accuracy_score(y_te_h1_b[known], y_pred_evid[known]):.3f} | F1_fail: {f1_ev_fail:.3f} | F1_succ: {f1_ev_succ:.3f} | N={known.sum()}')
else:
    print('EVIDENCE-PRESENCE: gold_entities_in_context flags not available')

# ── PER-LAYER TRAINING ────────────────────────────────────────────────────────
print('\n' + '=' * 120)
print('PROBE TRAINING PER LAYER')
print('F1_fail = pos_label=1 (Failure class); F1_succ = pos_label=0 (Success class)')
print('=' * 120)
print(f'{"Layer":<7}|{"AllHop Acc":>11}|{"F1_fail":>9}|{"F1_succ":>9}|{"AllHop AUC":>11}|{"H1 Acc":>7}|{"H1 F1_fail":>11}|{"H1 AUC":>8}')
print('-' * 80)

best_layer, best_auc = None, 0.0
layer_results = {}

for li in layer_indices:
    X_tr, y_tr, h_tr, _, _, _, _, _ = extract_data(train_ex, hs_dir, layer_idx=li)
    X_te, y_te, h_te, _, _, _, _, _ = extract_data(test_ex,  hs_dir, layer_idx=li)
    if len(y_tr) == 0 or len(y_te) == 0: continue

    # All-hop probe
    clf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    clf.fit(X_tr, y_tr)
    y_pred = clf.predict(X_te); y_prob = clf.predict_proba(X_te)[:, 1]
    acc = accuracy_score(y_te, y_pred)
    f1_fail = f1_score(y_te, y_pred, pos_label=1, zero_division=0)
    f1_succ = f1_score(y_te, y_pred, pos_label=0, zero_division=0)
    auc = roc_auc_score(y_te, y_prob)

    # Hop-1 probe
    h1_tr = h_tr == 0; h1_te = h_te == 0
    clf_h1 = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    clf_h1.fit(X_tr[h1_tr], y_tr[h1_tr])
    y_pred_h1 = clf_h1.predict(X_te[h1_te]); y_prob_h1 = clf_h1.predict_proba(X_te[h1_te])[:, 1]
    acc_h1 = accuracy_score(y_te[h1_te], y_pred_h1)
    f1_h1_fail = f1_score(y_te[h1_te], y_pred_h1, pos_label=1, zero_division=0)
    f1_h1_succ = f1_score(y_te[h1_te], y_pred_h1, pos_label=0, zero_division=0)
    auc_h1 = roc_auc_score(y_te[h1_te], y_prob_h1) if len(np.unique(y_te[h1_te])) > 1 else float('nan')

    print(f'Layer {li:<2}|{acc:>11.3f}|{f1_fail:>9.3f}|{f1_succ:>9.3f}|{auc:>11.3f}|{acc_h1:>7.3f}|{f1_h1_fail:>11.3f}|{auc_h1:>8.3f}')

    layer_results[li] = {
        'acc': acc, 'f1_fail': f1_fail, 'f1_succ': f1_succ, 'auc': auc,
        'acc_h1': acc_h1, 'f1_h1_fail': f1_h1_fail, 'f1_h1_succ': f1_h1_succ, 'auc_h1': auc_h1,
        'clf': clf, 'clf_h1': clf_h1
    }
    joblib.dump(clf_h1, probe_dir / f'hop1_probe_layer{li}.joblib')
    joblib.dump(clf,    probe_dir / f'allhop_probe_layer{li}.joblib')
    if auc > best_auc:
        best_auc = auc; best_layer = li

print(f'\nBEST LAYER (all-hop AUROC): Layer {best_layer}  (AUROC={best_auc:.3f})')
json.dump([ex['id'] for ex in train_ex], open(probe_dir / 'probe_train_ids.json', 'w'))
print(f'Probes saved to {probe_dir}')


## Step 7 — Extended Metrics on Best Layer

In [ ]:
import numpy as np
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline

li     = best_layer
clf_h1 = layer_results[li]['clf_h1']
clf    = layer_results[li]['clf']

X_tr, y_tr, h_tr, _, _, rc_tr, _, gic_tr = extract_data(train_ex, hs_dir, layer_idx=li)
X_te, y_te, h_te, _, _, rc_te, _, gic_te = extract_data(test_ex,  hs_dir, layer_idx=li)
h1_te = h_te == 0; h2_te = h_te == 1

# ── 5-FOLD GROUPED CV ─────────────────────────────────────────────────────────
print('\n' + '=' * 100)
print('5-FOLD GROUPED CROSS-VALIDATION (Hop-1, Best Layer) — Cross-fold variance')
print('=' * 100)
X_all_h1, y_all_h1, _, _, _, _, _, _ = extract_data(valid_ex, hs_dir, layer_idx=li, hop_filter=0)
groups_all = [
    ex.get('clean_pair_id', ex['id'])
    for ex in valid_ex
    for h in ex.get('hops', [])
    if h.get('hop_idx') == 1 and h.get('label') in (0, 1) and (hs_dir / f"{ex['id']}.pt").exists()
]
min_n = min(len(X_all_h1), len(groups_all))
X_all_h1, y_all_h1, groups_all = X_all_h1[:min_n], y_all_h1[:min_n], groups_all[:min_n]
g_enc = LabelEncoder().fit(groups_all).transform(groups_all)

cv_accs, cv_aucs, cv_f1_fails, cv_f1_succs = [], [], [], []
for tr_idx, te_idx in GroupKFold(n_splits=5).split(X_all_h1, y_all_h1, groups=g_enc):
    Xt, Xv = X_all_h1[tr_idx], X_all_h1[te_idx]
    yt, yv = y_all_h1[tr_idx], y_all_h1[te_idx]
    if len(np.unique(yt)) < 2 or len(np.unique(yv)) < 2: continue
    c = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
    c.fit(Xt, yt)
    yv_pred = c.predict(Xv)
    yv_prob = c.predict_proba(Xv)[:, 1]
    cv_accs.append(accuracy_score(yv, yv_pred))
    cv_aucs.append(roc_auc_score(yv, yv_prob))
    cv_f1_fails.append(f1_score(yv, yv_pred, pos_label=1, zero_division=0))
    cv_f1_succs.append(f1_score(yv, yv_pred, pos_label=0, zero_division=0))
print(f'Hop-1 @ Layer {li}:')
print(f'  CV Acc:      {np.mean(cv_accs):.3f} ± {np.std(cv_accs):.3f}')
print(f'  CV AUROC:    {np.mean(cv_aucs):.3f} ± {np.std(cv_aucs):.3f}')
print(f'  CV F1(fail): {np.mean(cv_f1_fails):.3f} ± {np.std(cv_f1_fails):.3f}  [pos_label=1, failure class]')
print(f'  CV F1(succ): {np.mean(cv_f1_succs):.3f} ± {np.std(cv_f1_succs):.3f}  [pos_label=0, success class]')

# ── CROSS-HOP GENERALIZATION ──────────────────────────────────────────────────
print('\n' + '=' * 100)
print('CROSS-HOP GENERALIZATION (Train Hop-1 → Test on Hop-2)')
print('=' * 100)
if h2_te.sum() > 0 and len(np.unique(y_te[h2_te])) > 1:
    y_pred_h2 = clf_h1.predict(X_te[h2_te])
    y_prob_h2 = clf_h1.predict_proba(X_te[h2_te])[:, 1]
    print(f'Hop-1 Probe on Hop-2 Test (N={h2_te.sum()}):')
    print(f'  Acc:      {accuracy_score(y_te[h2_te], y_pred_h2):.3f}')
    print(f'  AUROC:    {roc_auc_score(y_te[h2_te], y_prob_h2):.3f}')
    print(f'  F1(fail): {f1_score(y_te[h2_te], y_pred_h2, pos_label=1, zero_division=0):.3f}')
    print(f'  F1(succ): {f1_score(y_te[h2_te], y_pred_h2, pos_label=0, zero_division=0):.3f}')
    print('Finding: Cross-hop transfer drop shows probe representations are hop-specific.')
else:
    print('Hop-2 subset insufficient for cross-hop evaluation.')

# ── RETRIEVAL CONDITION BREAKDOWN ─────────────────────────────────────────────
print('\n' + '=' * 100)
print('HOP-1 PROBE PERFORMANCE BY RETRIEVAL CONDITION')
print('=' * 100)
print(f'{"Condition":<22}|{"N":>5}|{"Acc":>7}|{"F1_fail":>9}|{"F1_succ":>9}|{"AUROC":>8}|{"Fail%":>7}')
print('-' * 70)
rc_te_arr = np.array(rc_te)
for cond in ['both_present', 'only_hop1_passage', 'only_hop2_passage', 'neither',
             'only_p1', 'only_p2', 'unknown']:
    mask = h1_te & (rc_te_arr == cond)
    if mask.sum() < 3: continue
    yc, Xc = y_te[mask], X_te[mask]
    preds = clf_h1.predict(Xc); probs = clf_h1.predict_proba(Xc)[:, 1]
    f1_f  = f1_score(yc, preds, pos_label=1, zero_division=0)
    f1_s  = f1_score(yc, preds, pos_label=0, zero_division=0)
    auc_c = roc_auc_score(yc, probs) if len(np.unique(yc)) > 1 else float('nan')
    fail_pct = yc.mean() * 100  # label 1 = failure
    print(f'{cond:<22}|{mask.sum():>5}|{accuracy_score(yc, preds):>7.3f}|{f1_f:>9.3f}|{f1_s:>9.3f}|{auc_c:>8.3f}|{fail_pct:>6.1f}%')

# ── SHUFFLED LABEL CONTROL (AUROC) ────────────────────────────────────────────
print('\n' + '=' * 100)
print('SHUFFLED LABEL CONTROL — AUROC should be ~0.50')
print('=' * 100)
h1_tr = h_tr == 0
y_shuf = np.random.default_rng(42).permutation(y_tr[h1_tr])
clf_shuf = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
clf_shuf.fit(X_tr[h1_tr], y_shuf)
y_prob_shuf = clf_shuf.predict_proba(X_te[h1_te])[:, 1]
shuf_auc = roc_auc_score(y_te[h1_te], y_prob_shuf) if len(np.unique(y_te[h1_te])) > 1 else float('nan')
print(f'Shuffled AUROC: {shuf_auc:.3f}  (expect ~0.50, demonstrating learned representations are non-random)')
print('Note: Shuffled accuracy is biased by class imbalance; AUROC is the proper metric.')

# ── LAYER PLATEAU (CV spread, layers 8-14) ────────────────────────────────────
print('\n' + '=' * 100)
print('LAYER PLATEAU — Hop-1 AUROC CV spread across layers 8-14')
print('=' * 100)
print(f'{"Layer":<7}|{"CV AUROC":>9}|{"±std":>6}|{"CV Acc":>8}|{"±std":>6}')
print('-' * 40)
for li_b in [l for l in layer_indices if 8 <= l <= 14]:
    Xb, yb, _, _, _, _, _, _ = extract_data(valid_ex, hs_dir, layer_idx=li_b, hop_filter=0)
    nb = min(len(Xb), len(g_enc))
    Xb, yb, gb = Xb[:nb], yb[:nb], g_enc[:nb]
    cv_a, cv_au = [], []
    for tr_i, te_i in GroupKFold(n_splits=5).split(Xb, yb, groups=gb):
        Xt, Xv, yt, yv = Xb[tr_i], Xb[te_i], yb[tr_i], yb[te_i]
        if len(np.unique(yt)) < 2 or len(np.unique(yv)) < 2: continue
        c = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
        c.fit(Xt, yt)
        cv_a.append(accuracy_score(yv, c.predict(Xv)))
        cv_au.append(roc_auc_score(yv, c.predict_proba(Xv)[:, 1]))
    if cv_au:
        print(f'Layer {li_b:<2}|{np.mean(cv_au):>9.3f}|±{np.std(cv_au):.3f}|{np.mean(cv_a):>8.3f}|±{np.std(cv_a):.3f}')


## Step 8 — Hedging vs Confidently Wrong Failure Analysis

In [ ]:
import numpy as np

li     = best_layer
clf_h1 = layer_results[li]['clf_h1']
X_h1, y_h1, _, texts_h1, _, _, _, _ = extract_data(test_ex, hs_dir, layer_idx=li, hop_filter=0)

HEDGE_KEYWORDS = [
    'no information', 'no mention', 'not mentioned', 'no relevant information',
    'not found', 'cannot find', 'does not mention', 'unclear', 'no context',
    'the context does not', 'insufficient information', 'not specified'
]

fail_mask   = (y_h1 == 1)  # label 1 = true failure
X_fail      = X_h1[fail_mask]
texts_fail  = np.array(texts_h1)[fail_mask]
y_pred_fail = clf_h1.predict(X_fail)
probs_fail  = clf_h1.predict_proba(X_fail)[:, 1]  # P(failure)

is_hedging = np.array([any(k in t.lower() for k in HEDGE_KEYWORDS) for t in texts_fail])

print('=' * 100)
print(f'HEDGING vs CONFIDENTLY WRONG FAILURE ANALYSIS (Hop-1, Layer {li})')
print('=' * 100)
print(f'Total hop-1 failures (label=1): {fail_mask.sum()}')
n_hedge = is_hedging.sum(); n_conf = (~is_hedging).sum()
if n_hedge > 0:
    caught = (y_pred_fail[is_hedging] == 1).mean()  # probe predicts failure
    print(f'  Hedging failures:           {n_hedge:3d} | Probe detection rate: {caught:.3f} | Mean P(fail): {probs_fail[is_hedging].mean():.3f}')
else:
    print('  Hedging failures:           0')
if n_conf > 0:
    caught = (y_pred_fail[~is_hedging] == 1).mean()  # probe predicts failure
    print(f'  Confidently Wrong failures: {n_conf:3d} | Probe detection rate: {caught:.3f} | Mean P(fail): {probs_fail[~is_hedging].mean():.3f}')
else:
    print('  Confidently Wrong failures: 0')


## Step 9 — Qualitative Analysis

In [ ]:
import numpy as np, torch, json

li     = best_layer
clf_h1 = layer_results[li]['clf_h1']

qual_results = []
for ex in test_ex:
    pt_path = hs_dir / f"{ex['id']}.pt"
    if not pt_path.exists(): continue
    data   = torch.load(pt_path, weights_only=False)
    pooled = data['pooled']
    ex_gic = ex.get('gold_entities_in_context', [])
    for hop in ex.get('hops', []):
        if hop.get('hop_idx') != 1 or hop.get('label') not in (0, 1): continue
        h_idx = hop['hop_idx'] - 1
        if h_idx >= pooled.shape[1]: continue
        vec       = np.nan_to_num(pooled[li, h_idx].detach().numpy())
        prob_fail = float(clf_h1.predict_proba([vec])[0][1])  # P(failure / label=1)
        gold_in_ctx = ex_gic[h_idx] if h_idx < len(ex_gic) else None
        qual_results.append({
            'prob_fail':       prob_fail,
            'true_label':      hop['label'],  # 0=success, 1=failure
            'question':        ex.get('question', ''),
            'gold_entity':     hop.get('bridging_entity_gold', ''),
            'generated_text':  hop.get('text', ''),
            'retrieval_cond':  ex.get('retrieval_condition', 'unknown'),
            'gold_in_context': gold_in_ctx,
        })

qual_results.sort(key=lambda x: x['prob_fail'], reverse=True)

print('=' * 100)
print(f'TOP 10 MOST CONFIDENT FAILURE PREDICTIONS (Layer {li})')
print('=' * 100)
for i, r in enumerate(qual_results[:10]):
    label_str = 'TP (Real Failure)' if r['true_label'] == 1 else 'FP (False Alarm / Actually Succeeded)'
    print(f"\n[{i+1}] {label_str} | P(fail)={r['prob_fail']:.3f} | retrieval={r['retrieval_cond']} | gold_in_ctx={r['gold_in_context']}")
    print(f"  Q:    {r['question']}")
    print(f"  Gold: {r['gold_entity']}")
    print(f"  Hop:  {r['generated_text'][:200]}")

print('\n' + '=' * 100)
print('TRUE REASONING FAILURES (Gold entity IN context, model still failed / label=1)')
print('=' * 100)
reasoning_fails = [r for r in qual_results if r['true_label'] == 1 and r['gold_in_context'] is True]
print(f'Total True Reasoning Failures: {len(reasoning_fails)}')
for i, r in enumerate(reasoning_fails[:5]):
    print(f"\n[{i+1}] P(fail)={r['prob_fail']:.3f} | retrieval={r['retrieval_cond']}")
    print(f"  Q:    {r['question']}")
    print(f"  Gold: {r['gold_entity']}")
    print(f"  Hop:  {r['generated_text'][:200]}")

print('\n' + '=' * 100)
print('FALSE NEGATIVES (Actual Failures Probe Missed — P(fail) < 0.5)')
print('=' * 100)
false_negs = sorted([r for r in qual_results if r['true_label'] == 1 and r['prob_fail'] < 0.5],
                    key=lambda x: x['prob_fail'])
print(f'Total False Negatives: {len(false_negs)}')
for i, r in enumerate(false_negs[:5]):
    print(f"\n[{i+1}] P(fail)={r['prob_fail']:.3f} | retrieval={r['retrieval_cond']} | gold_in_ctx={r['gold_in_context']}")
    print(f"  Q:    {r['question']}")
    print(f"  Gold: {r['gold_entity']}")
    print(f"  Hop:  {r['generated_text'][:200]}")

json.dump(qual_results, open('/kaggle/working/qualitative_results.json', 'w'), indent=2, default=str)
print(f'\nSaved {len(qual_results)} qualitative results to /kaggle/working/qualitative_results.json')


## Step 10 — Upload Probes & Results to HuggingFace

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

api = HfApi(token=HF_TOKEN)
probe_dir = Path('/kaggle/working/probes_tfidf_rag')
try:
    api.create_branch(repo_id=HF_REPO_ID, branch=HF_REVISION, repo_type='dataset', exist_ok=True)
except Exception as e:
    print(f'Branch note: {e}')

print(f'Uploading probes to {HF_REPO_ID} [{HF_REVISION}] / {PROBE_SUBFOLDER} ...')
uploaded = 0
for f in sorted(probe_dir.iterdir()):
    api.upload_file(
        path_or_fileobj=str(f),
        path_in_repo=f'{PROBE_SUBFOLDER}/{f.name}',
        repo_id=HF_REPO_ID, revision=HF_REVISION, repo_type='dataset',
        commit_message=f'Probe: {f.name}'
    )
    uploaded += 1; print(f'  ok {f.name}')

api.upload_file(
    path_or_fileobj='/kaggle/working/qualitative_results.json',
    path_in_repo=f'{PROBE_SUBFOLDER}/qualitative_results.json',
    repo_id=HF_REPO_ID, revision=HF_REVISION, repo_type='dataset',
    commit_message='Qualitative results'
)
print(f'Uploaded {uploaded} probe files + qualitative_results.json')


## Step 11 — Upload labeled.jsonl & wikidata_aliases.json (Permanent Cache)

In [ ]:
from huggingface_hub import HfApi
from pathlib import Path

api = HfApi(token=HF_TOKEN)
labeled_local = Path('/kaggle/working/data/raw/2wikimultihopqa/labeled.jsonl')
aliases_local = Path('/kaggle/working/data/raw/wikidata_aliases.json')

if labeled_local.exists():
    api.upload_file(
        path_or_fileobj=str(labeled_local),
        path_in_repo=f'{HF_SUBFOLDER}/data/raw/2wikimultihopqa/labeled.jsonl',
        repo_id=HF_REPO_ID, revision=HF_REVISION, repo_type='dataset',
        commit_message='Save labeled.jsonl (permanent cache)'
    )
    print('ok labeled.jsonl uploaded to HF')
else:
    print('labeled.jsonl not found on disk')

if aliases_local.exists():
    api.upload_file(
        path_or_fileobj=str(aliases_local),
        path_in_repo='wikidata_aliases.json',
        repo_id=HF_REPO_ID, revision=HF_REVISION, repo_type='dataset',
        commit_message='Cache wikidata_aliases.json (saves 1hr on future runs)'
    )
    print('ok wikidata_aliases.json uploaded to HF')
else:
    print('wikidata_aliases.json not found on disk')

print('\nPipeline complete! All files and probes safely uploaded.')
